# MODUL PRAKTIKUM DATA MINING
## Association Rule Mining (Market Basket Analysis)

Notebook ini merupakan file terpisah untuk mengerjakan **Tugas 3 (Bagian 5.3)** pada Modul Praktikum Data Mining – Association Rule Mining (Market Basket Analysis), 2026.

### Dataset yang digunakan
Dataset transaksi yang digunakan tetap sama dengan notebook praktikum utama, yaitu `transactions_arm_praktikum.csv`, berisi 60 transaksi dengan 12 jenis item.

### Alur pengerjaan
1. Menjalankan algoritma Apriori dan FP-Growth dengan `min_support = 0.05`.
2. Mencatat jumlah frequent itemset dan waktu eksekusi masing-masing algoritma.
3. Membandingkan hasil eksperimen dengan Bagian 4.8 yang menggunakan `min_support = 0.15`.
4. Menganalisis perbedaan waktu eksekusi dan keunggulan kedua algoritma.

Seluruh proses menggunakan dataset dan pustaka yang sama dengan praktikum utama. Perubahan dilakukan pada nilai `min_support` dan bagian perbandingan waktu eksekusi sesuai instruksi Tugas 3.

## Persiapan Data dan Pustaka

Tahap persiapan dilakukan dengan mengimpor pustaka yang dibutuhkan dan membaca dataset transaksi yang sama dengan praktikum utama. Data transaksi kemudian diubah menjadi matriks one-hot menggunakan `TransactionEncoder` agar dapat diproses oleh algoritma Apriori dan FP-Growth.

In [1]:
%pip install -q mlxtend

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth

# Membaca dataset transaksi mentah
data = pd.read_csv('transactions_arm_praktikum.csv')

# transactions: list berisi transaksi (daftar item per transaksi)
transactions = [items.split(', ') for items in data['Items']]

print('Jumlah transaksi:', len(transactions))
for i, t in enumerate(transactions[:5], 1):
    print(f'T{i}: {t}')

Jumlah transaksi: 60
T1: ['Gula', 'Kopi', 'Susu']
T2: ['Gula', 'Kopi', 'Susu']
T3: ['Mentega', 'Roti', 'Susu', 'Telur']
T4: ['Mie Instan']
T5: ['Keju', 'Roti', 'Susu', 'Telur']


In [3]:
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)

print('Ukuran matriks one-hot:', df.shape)
print(df.iloc[:5, :6])

Ukuran matriks one-hot: (60, 12)
    Gula  Kecap   Keju   Kopi  Mentega  Mie Instan
0   True  False  False   True    False       False
1   True  False  False   True    False       False
2  False  False  False  False     True       False
3  False  False  False  False    False        True
4  False  False   True  False    False       False


### 1. Ulangi eksperimen Bagian 4.8 dengan `min_support = 0.05`.

Pada tahap ini, algoritma Apriori dan FP-Growth dijalankan kembali menggunakan nilai `min_support = 0.05`, lebih rendah dibandingkan nilai 0.15 pada Bagian 4.8.

Penurunan nilai ambang batas ini bertujuan untuk menemukan lebih banyak frequent itemset, sehingga perbedaan waktu eksekusi kedua algoritma dapat diamati pada jumlah itemset yang lebih banyak.

In [4]:
MIN_SUP = 0.05

# Mengukur waktu eksekusi Apriori
t0 = time.perf_counter()
freq_apriori_005 = apriori(
    df,
    min_support=MIN_SUP,
    use_colnames=True
)
t_apriori_005 = time.perf_counter() - t0

# Mengukur waktu eksekusi FP-Growth
t0 = time.perf_counter()
freq_fpgrowth_005 = fpgrowth(
    df,
    min_support=MIN_SUP,
    use_colnames=True
)
t_fpgrowth_005 = time.perf_counter() - t0

# Membandingkan hasil frequent itemset
set_apriori_005 = set(
    frozenset(s) for s in freq_apriori_005['itemsets']
)

set_fpgrowth_005 = set(
    frozenset(s) for s in freq_fpgrowth_005['itemsets']
)

print('Hasil Eksperimen min_support = 0.05')
print(f'Apriori  : {len(freq_apriori_005)} itemset, {t_apriori_005*1000:.2f} ms')
print(f'FP-Growth: {len(freq_fpgrowth_005)} itemset, {t_fpgrowth_005*1000:.2f} ms')
print('Itemset identik?', set_apriori_005 == set_fpgrowth_005)

Hasil Eksperimen min_support = 0.05
Apriori  : 42 itemset, 6.37 ms
FP-Growth: 42 itemset, 5.05 ms
Itemset identik? True


### 2. Perbandingan jumlah itemset dan waktu eksekusi.

Untuk mengetahui pengaruh penurunan nilai `min_support`, hasil eksperimen pada 0.05 dibandingkan dengan hasil Bagian 4.8 yang menggunakan `min_support = 0.15`.

Agar perbandingan waktu lebih konsisten, kedua algoritma dijalankan kembali pada kedua nilai ambang batas dalam lingkungan eksekusi yang sama.

In [5]:
hasil_perbandingan = []

for s in [0.15, 0.05]:

    # Apriori
    t0 = time.perf_counter()
    fi_apriori = apriori(
        df,
        min_support=s,
        use_colnames=True
    )
    waktu_apriori = time.perf_counter() - t0

    # FP-Growth
    t0 = time.perf_counter()
    fi_fpgrowth = fpgrowth(
        df,
        min_support=s,
        use_colnames=True
    )
    waktu_fpgrowth = time.perf_counter() - t0

    hasil_perbandingan.append({
        'min_support': s,
        'Apriori Itemset': len(fi_apriori),
        'FP-Growth Itemset': len(fi_fpgrowth),
        'Apriori (ms)': waktu_apriori * 1000,
        'FP-Growth (ms)': waktu_fpgrowth * 1000
    })

hasil_df = pd.DataFrame(hasil_perbandingan)

print('Perbandingan Apriori dan FP-Growth')
display(hasil_df.round(3))

Perbandingan Apriori dan FP-Growth


,min_support,Apriori Itemset,FP-Growth Itemset,Apriori (ms),FP-Growth (ms)
0,0.15,17,17,5.624,2.192
1,0.05,42,42,5.827,2.882


In [6]:
selisih = []

for _, row in hasil_df.iterrows():

    beda = abs(
        row['Apriori (ms)'] - row['FP-Growth (ms)']
    )

    rasio = (
        row['Apriori (ms)'] / row['FP-Growth (ms)']
        if row['FP-Growth (ms)'] > 0 else np.nan
    )

    selisih.append({
        'min_support': row['min_support'],
        'Selisih Waktu (ms)': beda,
        'Rasio Apriori/FP-Growth': rasio
    })

selisih_df = pd.DataFrame(selisih)

print('Selisih Waktu Eksekusi')
display(selisih_df.round(3))

# Membandingkan perubahan selisih waktu
selisih_015 = selisih_df.loc[
    selisih_df['min_support'] == 0.15,
    'Selisih Waktu (ms)'
].iloc[0]

selisih_005 = selisih_df.loc[
    selisih_df['min_support'] == 0.05,
    'Selisih Waktu (ms)'
].iloc[0]

print('\nPerubahan selisih waktu:')
print(f'min_support 0.15: {selisih_015:.3f} ms')
print(f'min_support 0.05: {selisih_005:.3f} ms')

if selisih_005 > selisih_015:
    print('Selisih waktu semakin besar pada min_support = 0.05.')
elif selisih_005 < selisih_015:
    print('Selisih waktu semakin kecil pada min_support = 0.05.')
else:
    print('Selisih waktu relatif sama pada kedua ambang batas.')

Selisih Waktu Eksekusi


,min_support,Selisih Waktu (ms),Rasio Apriori/FP-Growth
0,0.15,3.432,2.566
1,0.05,2.945,2.022



Perubahan selisih waktu:
min_support 0.15: 3.432 ms
min_support 0.05: 2.945 ms
Selisih waktu semakin kecil pada min_support = 0.05.


**Analisis hasil:**

Berdasarkan hasil eksperimen, penggunaan `min_support = 0.05` menghasilkan jumlah frequent itemset yang lebih banyak dibandingkan `min_support = 0.15`. Hal ini terjadi karena ambang batas yang lebih rendah memungkinkan lebih banyak kombinasi item dengan frekuensi kemunculan yang lebih sedikit untuk memenuhi persyaratan frequent itemset.

Pada kedua nilai `min_support`, Apriori dan FP-Growth menghasilkan jumlah frequent itemset yang sama. Hal ini menunjukkan bahwa kedua algoritma menggunakan prinsip perhitungan frequent itemset yang sama, tetapi memiliki strategi pencarian yang berbeda.

Dari segi waktu eksekusi, perbedaan performa kedua algoritma dapat dilihat melalui hasil pengukuran dan grafik perbandingan. Algoritma dengan waktu eksekusi lebih rendah menunjukkan proses pencarian frequent itemset yang lebih cepat pada pengujian tersebut.

Perubahan selisih waktu antara `min_support = 0.15` dan `min_support = 0.05` dapat digunakan untuk melihat apakah keunggulan FP-Growth semakin terlihat ketika jumlah frequent itemset meningkat.

### 3. Apakah keunggulan FP-Growth semakin terlihat ketika jumlah itemset yang dicari semakin banyak?

FP-Growth memiliki keunggulan karena tidak menghasilkan kandidat itemset secara eksplisit seperti Apriori. Algoritma ini menggunakan struktur FP-Tree untuk menyimpan dan memadatkan informasi transaksi, kemudian menemukan frequent pattern melalui proses penambangan pada struktur tersebut.

Sebaliknya, Apriori menggunakan proses pembentukan kandidat itemset secara bertahap dan melakukan pemindaian data berulang kali pada setiap tingkat itemset. Ketika nilai `min_support` diturunkan, semakin banyak kombinasi item yang harus diperiksa sehingga proses pencarian dapat menjadi lebih berat.

Dengan demikian, secara teori, keunggulan FP-Growth cenderung semakin terlihat pada dataset yang lebih besar dan memiliki banyak frequent itemset. Namun, pada dataset praktikum yang hanya terdiri atas 60 transaksi dan 12 jenis item, perbedaan waktu eksekusi belum tentu menunjukkan peningkatan keunggulan yang besar. Hal ini karena ukuran dataset masih relatif kecil sehingga waktu pemrosesan kedua algoritma sama-sama singkat.

## Kesimpulan

Berdasarkan eksperimen perbandingan algoritma Apriori dan FP-Growth pada `min_support = 0.05`, dapat diketahui bahwa penurunan ambang batas minimum support memungkinkan lebih banyak frequent itemset ditemukan dibandingkan penggunaan `min_support = 0.15`.

Kedua algoritma menghasilkan jumlah frequent itemset yang identik, tetapi memiliki waktu eksekusi yang berbeda. Perbedaan tersebut dipengaruhi oleh strategi pencarian masing-masing algoritma. Apriori melakukan pembentukan kandidat itemset secara bertahap, sedangkan FP-Growth menggunakan struktur FP-Tree untuk menemukan pola tanpa menghasilkan kandidat secara eksplisit.

Secara teori, FP-Growth lebih efisien ketika berhadapan dengan dataset besar dan jumlah frequent itemset yang banyak. Namun, pada dataset praktikum yang relatif kecil, keunggulan tersebut belum tentu terlihat secara signifikan. Oleh karena itu, pemilihan algoritma perlu mempertimbangkan ukuran dataset, jumlah item, dan hasil pengukuran waktu eksekusi.